In [1]:
from camel.agents import ChatAgent
from camel.types import ModelType
from datasets import load_dataset
from librarian.model import create_openai_model
from librarian.evaluation import SimpleQAGrader
from librarian.schema import LibrarianResponse, PlainResponse, CoTResponse
from librarian.prompt import PLAIN_PROMPT, COT_PROMPT, LIBRARIAN_PROMPT

/home/yuan/.cache/pypoetry/virtualenvs/librarian-yhAyT7eh-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("basicv8vc/SimpleQA")

In [3]:
librarian_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=LIBRARIAN_PROMPT)
plain_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=PLAIN_PROMPT)
cot_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=COT_PROMPT)
evaluation_agent = SimpleQAGrader(ChatAgent(model=create_openai_model(ModelType.GPT_4O)))

In [4]:
librarian_responses = []
plain_responses = []
cot_responses = []
librarian_grades = []
plain_grades = []
cot_grades = []

for dp in list(dataset["test"])[8:10]:
    
    # reset all agents
    librarian_agent.reset()
    plain_agent.reset()
    cot_agent.reset()
    
    print("#### Question ####\n")
    print(dp["problem"])
    
    print("\n#### Librarian Agent Answer ####\n")
    librarian_response = librarian_agent.step(f"Question: {dp['problem']}", response_format=LibrarianResponse)
    librarian_response = eval(librarian_response.msgs[0].content)
    librarian_responses.append(librarian_response)
    print("knowledge:", librarian_response["knowledge"], "\n")
    print("reasoning:", librarian_response["reasoning"], "\n")
    print("answer:", librarian_response["answer"])
    
    print("\n#### Plain Agent Answer ####\n")
    plain_response = plain_agent.step(dp["problem"], response_format=PlainResponse)
    plain_response = eval(plain_response.msgs[0].content)
    plain_responses.append(plain_response)
    print("answer:", plain_response["answer"])
    
    print("\n#### CoT Agent Answer ####\n")
    cot_response = cot_agent.step(dp["problem"], response_format=CoTResponse)
    cot_response = eval(cot_response.msgs[0].content)
    cot_responses.append(cot_response)
    print("reasoning:", cot_response["reasoning"], "\n")
    print("answer:", cot_response["answer"])
    
    print("\n#### Gold Answer ####\n")
    print(dp["answer"])
    
    print("\n#### Grades ####\n")
    
    librarian_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], librarian_response["answer"]))["grade"]
    plain_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], plain_response["answer"]))["grade"]
    cot_grade = eval(evaluation_agent.grade(dp['problem'], dp['answer'], cot_response["answer"]))["grade"]
    
    librarian_grades.append(librarian_grade)
    plain_grades.append(plain_grade)
    cot_grades.append(cot_grade)
    
    print("Librarian Agent Grade:", librarian_grade)
    print("Plain Agent Grade:", plain_grade)
    print("CoT Agent Grade:", cot_grade)
    
    print("\n#### Cumulative Results ###\n")
    
    print("Librarian Agent Cumulative Correct:", librarian_grades.count("CORRECT"), "/", len(librarian_grades))
    print("Plain Agent Cumulative Correct:", plain_grades.count("CORRECT"), "/", len(plain_grades))
    print("CoT Agent Cumulative Correct:", cot_grades.count("CORRECT"), "/", len(cot_grades))
    
    print("\n")
    print("---------")
    print("\n")

#### Question ####

What is the name of the former Prime Minister of Iceland who worked as a cabin crew member until 1971?

#### Librarian Agent Answer ####

knowledge: [',{"source":"Wikipedia","timestamp":"2023-10-01","fact":"Jóhanna Sigurðardóttir, who served as the Prime Minister of Iceland from 2009 to 2013, worked as a flight attendant for Loftleiðir Icelandic Airlines from 1962 to 1971."}'] 

reasoning: ['The question asks for the name of a former Prime Minister of Iceland who worked as a cabin crew member until 1971.', 'According to the retrieved knowledge, Jóhanna Sigurðardóttir worked as a flight attendant until 1971 and later became the Prime Minister of Iceland.'] 

answer: Jóhanna Sigurðardóttir

#### Plain Agent Answer ####

answer: Jóhanna Sigurðardóttir

#### CoT Agent Answer ####

reasoning: ['We need to identify a former Prime Minister of Iceland who had a career as a cabin crew member until 1971.', 'This suggests we are looking for someone who became Prime Minister af

{'librarian': ['NOT_ATTEMPTED'], 'plain': ['INCORRECT'], 'cot': ['INCORRECT']}